# Feature Engineering - Notebook 03

**Objectif** : Créer des variables pertinentes pour améliorer la capacité prédictive du modèle de churn.

**Stratégie appliquée** :
1. **Flags de risque** : Identifier les segments à fort risque de départ
2. **Binning** : Discrétiser les variables continues pour capturer des effets non-linéaires
3. **Variables d'interaction** : Combiner des features identifiées comme discriminantes
4. **Features métier** : Créer des ratios et indicateurs d'engagement client

Les features créées sont basées sur les insights de l'analyse exploratoire (notebook 02) et visent à capturer les comportements clients complexes.

## 1. Import et vérification

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re 
import warnings
from itertools import combinations

pd.set_option('display.max_rows', 1000)
warnings.filterwarnings('ignore')
df = pd.read_csv('../data/preprocessed/02_churn_records.csv')
df = df.convert_dtypes()

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   row_number          10000 non-null  Int64  
 1   customer_id         10000 non-null  Int64  
 2   surname             10000 non-null  string 
 3   credit_score        10000 non-null  Int64  
 4   age                 10000 non-null  Int64  
 5   tenure              10000 non-null  Int64  
 6   balance             10000 non-null  Float64
 7   num_of_products     10000 non-null  Int64  
 8   has_cr_card         10000 non-null  Int64  
 9   is_active_member    10000 non-null  Int64  
 10  estimated_salary    10000 non-null  Float64
 11  exited              10000 non-null  Int64  
 12  complain            10000 non-null  Int64  
 13  satisfaction_score  10000 non-null  Int64  
 14  point_earned        10000 non-null  Int64  
 15  geography_France    10000 non-null  boolean
 16  geogr

## 2. Création des flags de risque

Création de variables booléennes pour identifier les profils à risque élevé, basés sur les observations de l'EDA :
- **low_credit_score_risk** : Score de crédit ≤ 450 (associé à un taux de churn élevé)
- **medium_balance_risk** : Solde entre 100k-140k (zone d'instabilité observée)
- **low_point_earned_risk** : Points de fidélité très faibles (≤ 200)
- **open_account_bf_major** : Compte ouvert avant la majorité (potentiel changement de comportement)

In [5]:
medium_balance = (df['balance'] >= 100000) &  (df['balance'] <= 140000)
low_point_earned = (df['point_earned'] <= 200)
low_credit_score = (df['credit_score'] <= 450)
open_account_bf_major = (df['age'] - df['tenure'] < 18)

df['medium_balance_risk'] = medium_balance
df['low_credit_score_risk'] = low_credit_score
df['low_point_earned_risk'] = low_point_earned
df['open_account_bf_major'] = open_account_bf_major

print("Résumé des flags crées :\n")
print(f"low_credit_score_risk : {(df['low_credit_score_risk'] == True).sum()} valeurs à True")
print(f"medium_balance_risk : {(df['medium_balance_risk'] == True).sum()} valeurs à True")
print(f"low_point_earned_risk : {(df['low_point_earned_risk'] == True).sum()} valeurs à True")
print(f"open_account_bf_major : {(df['open_account_bf_major'] == True).sum()} valeurs à True")

Résumé des flags crées :

low_credit_score_risk : 189 valeurs à True
medium_balance_risk : 3250 valeurs à True
low_point_earned_risk : 2 valeurs à True
open_account_bf_major : 322 valeurs à True


## 3. Binning de l'âge

Discrétisation de la variable `age` en 4 catégories pour capturer la relation non-linéaire avec le churn :
- **18-30 ans** : Jeunes clients (faible churn)
- **30-45 ans** : Adultes actifs (churn modéré)
- **45-65 ans** : Seniors actifs (churn très élevé - 49.9%)
- **65+ ans** : Retraités (churn modéré)

Cette transformation permet au modèle de mieux capturer l'effet "pic de churn" chez les 45-65 ans.

In [6]:
df['age_bin'] = pd.cut(df['age'], bins=[17,30,45,65,100], labels=[1, 2, 3, 4])

print("Age binning :")
print(df.groupby('age_bin')['exited'].mean().sort_values(ascending=False))

Age binning :
age_bin
3    0.499188
2    0.157575
4    0.132576
1    0.075203
Name: exited, dtype: Float64


## 4. Variables d'interaction

Création de features combinant plusieurs variables pour capturer des effets synergiques. Ces interactions permettent de modéliser des comportements complexes qui ne seraient pas détectables en examinant les variables individuellement.

### 4.1 Interactions entre variables à fort taux de churn

Analyse combinatoire pour identifier les paires de conditions associées aux taux de churn les plus élevés. Les règles d'association révèlent que certaines combinaisons amplifient significativement le risque de départ.

In [7]:
masks = {
    'product_1': df['num_of_products'] == 1,
    'germany': df['geography_Germany'] == 1,
    'female': df['gender_Female'] == 1,
    'active_member': df['is_active_member'] == 1
}

results = []

for (name1, mask1), (name2, mask2) in combinations(masks.items(), 2):
    
    combined_mask = mask1 & mask2
    exited_rate = df.loc[combined_mask, 'exited'].mean()
    count = combined_mask.sum()

    results.append({
        'rule_1': name1,
        'rule_2': name2,
        'count': count,
        'exited_rate': exited_rate
    })

df_result = pd.DataFrame(results)
print(df_result.dropna().head(30).sort_values(by=['exited_rate'], ascending=False))

      rule_1         rule_2  count  exited_rate
0  product_1        germany   1349     0.428466
3    germany         female   1193     0.375524
1  product_1         female   2296     0.331882
4    germany  active_member   1248     0.237179
2  product_1  active_member   2563     0.189231
5     female  active_member   2284     0.181261


In [8]:
df["product_1_AND_germany"] = masks['product_1'] & masks['germany']
df["germany_AND_female"] = masks['germany'] & masks['female']
df["product_1_AND_female"] = masks['product_1'] & masks['female']

print("Variables d'intéractions :")
print(f"product_1_AND_germany : {(df['product_1_AND_germany'] == True).sum()} valeurs à True")
print(f"germany_AND_female : {(df['germany_AND_female'] == True).sum()} valeurs à True")
print(f"product_1_AND_female : {(df['product_1_AND_female'] == True).sum()} valeurs à True")

Variables d'intéractions :
product_1_AND_germany : 1349 valeurs à True
germany_AND_female : 1193 valeurs à True
product_1_AND_female : 2296 valeurs à True


### 4.2 Features métier et ratios

Création de variables basées sur la logique métier et les comportements clients observés :
- **Engagement produit** : Ratio entre nombre de produits et activité
- **Ratios financiers** : Balance par rapport au salaire, au nombre de produits, à l'ancienneté
- **Segments comportementaux** : Combinaisons de caractéristiques (âge, statut, balance) identifiant des profils spécifiques

Ces features capturent des dynamiques complexes comme "client inactif avec solde élevé" ou "senior avec balance importante".

In [9]:
df['product_1_inactive'] = (
    (df['num_of_products'] == 1) & 
    (df['is_active_member'] == 0)
).astype(int)

df['product_engagement_score'] = (
    df['num_of_products'] * df['is_active_member']
)

df['balance_to_salary_ratio'] = (
    df['balance'] / (df['estimated_salary'] + 1)
)

df['balance_per_product'] = (
    df['balance'] / (df['num_of_products'] + 1)
)

df['high_balance_inactive'] = (
    (df['balance'] > df['balance'].quantile(0.75)) & 
    (df['is_active_member'] == 0)
).astype(int)

df['young_explorer'] = (
    (df['age'] < 30) & 
    (df['tenure'] <= 2) & 
    (df['num_of_products'] == 1)
).astype(int)

df['balance_per_tenure_year'] = (
    df['balance'] / (df['tenure'] + 1)
)

df['senior_high_balance'] = (
    (df['age'] >= 50) & 
    (df['balance'] > df['balance'].quantile(0.75))
).astype(int)

df['premium_card_active'] = (
    (df['card_type_GOLD'] | df['card_type_PLATINUM'] | df['card_type_DIAMOND']) & 
    (df['is_active_member'] == 1)
).astype(int)

df['satisfaction_engagement'] = (
    df['satisfaction_score'] * df['is_active_member']
)

df['engagement_score'] = (
    df['is_active_member'] * 2 + 
    (df['balance'] > 0).astype(int) +  
    (df['tenure'] >= 3).astype(int)  
)

print("Features supplémentaires créees:")
print(f"product_1_inactive : {(df['product_1_inactive'] == True).sum()} valeurs à True")
print(f"high_balance_inactive : {(df['high_balance_inactive'] == True).sum()} valeurs à True")
print(f"young_explorer : {(df['young_explorer'] == True).sum()} valeurs à True")
print(f"senior_high_balance : {(df['senior_high_balance'] == True).sum()} valeurs à True")
print(f"premium_card_active : {(df['premium_card_active'] == True).sum()} valeurs à True\n")
print(df[[
    'product_engagement_score', 
    'balance_to_salary_ratio', 
    'balance_per_product', 
    'balance_per_tenure_year', 
    'satisfaction_engagement',
    'engagement_score'
]].describe().T[['min', 'mean', '50%', 'max']].round(2))

Features supplémentaires créees:
product_1_inactive : 2521 valeurs à True
high_balance_inactive : 1247 valeurs à True
young_explorer : 175 valeurs à True
senior_high_balance : 345 valeurs à True
premium_card_active : 3849 valeurs à True

                          min      mean       50%        max
product_engagement_score  0.0      0.79       1.0        4.0
balance_to_salary_ratio   0.0      3.79      0.75    9770.88
balance_per_product       0.0  33603.67  38452.86  119193.78
balance_per_tenure_year   0.0  18730.91  13280.98   197041.8
satisfaction_engagement   0.0      1.56       1.0        5.0
engagement_score          0.0      2.42       2.0        4.0


## 5. Nettoyage final du dataset

Suppression des features redondantes ou inutiles pour la modélisation :
- **Variables intégrées dans d'autres features** : `complain`, `satisfaction_score`, `tenure`, `point_earned`
- **Identifiants non-prédictifs** : `row_number`, `customer_id`, `surname`
- **Variables encodées** : types de cartes (déjà capturés dans les interactions)
- **Variables à faible pouvoir prédictif** : `has_cr_card`, `estimated_salary`, `credit_score`

Le dataset final contient **32 features** optimisées pour la phase de modélisation.

In [10]:
df = df.drop(columns=[
    'complain', 
    'satisfaction_score', 
    'tenure', 
    'point_earned', 
    'card_type', 
    'estimated_salary',
    'credit_score',
    'row_number',
    'customer_id',
    'surname',
    'card_type_DIAMOND',
    'card_type_PLATINUM',
    'card_type_GOLD',
    'card_type_SILVER',
    'has_cr_card'
], errors='ignore')

print("Features supprimées")
print(f"Dimensions du dataset : {df.shape}")
print(df.info())

Features supprimées
Dimensions du dataset : (10000, 29)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   age                       10000 non-null  Int64   
 1   balance                   10000 non-null  Float64 
 2   num_of_products           10000 non-null  Int64   
 3   is_active_member          10000 non-null  Int64   
 4   exited                    10000 non-null  Int64   
 5   geography_France          10000 non-null  boolean 
 6   geography_Germany         10000 non-null  boolean 
 7   geography_Spain           10000 non-null  boolean 
 8   gender_Female             10000 non-null  boolean 
 9   gender_Male               10000 non-null  boolean 
 10  medium_balance_risk       10000 non-null  boolean 
 11  low_credit_score_risk     10000 non-null  boolean 
 12  low_point_earned_risk     10000 non-null  boole

## 6. Évaluation de l'impact des features

Analyse de corrélation avec la variable cible `exited` pour évaluer le pouvoir prédictif de chaque feature créée.

**Top 5 features positivement corrélées** (favorisent le churn) :
- `age` et `age_bin` : L'âge reste le prédicteur le plus fort
- `senior_inactive` : Combinaison âge + inactivité particulièrement discriminante
- `product_1_inactive` : Clients avec un seul produit et inactifs
- `product_1_AND_germany` : Interaction géographique forte

**Top 5 features négativement corrélées** (réduisent le churn) :
- `is_active_member` : L'engagement reste protecteur
- `product_engagement_score` : Score combinant produits et activité
- `satisfaction_engagement` : Satisfaction × activité
- `premium_card_active` : Clients premium actifs plus fidèles
- `geography_France` : Marché plus stable

In [11]:
df_corr = df.copy()

corr_matrix = df_corr.corr()['exited'].sort_values(ascending=False)

for label, value in corr_matrix.drop('exited').items():
    print(label + " " * ((30 if value > 0 else 29) -len(label)) + f"{value:.2f}" + "  " + "█" * int(abs(value) * 100))

age                           0.29  ████████████████████████████
age_bin                       0.27  ███████████████████████████
product_1_inactive            0.23  ███████████████████████
product_1_AND_germany         0.22  ██████████████████████
product_1_AND_female          0.17  █████████████████
geography_Germany             0.17  █████████████████
germany_AND_female            0.16  ███████████████
senior_high_balance           0.14  █████████████
balance                       0.12  ███████████
balance_per_product           0.11  ██████████
gender_Female                 0.11  ██████████
medium_balance_risk           0.10  ██████████
high_balance_inactive         0.09  █████████
balance_per_tenure_year       0.08  ████████
low_credit_score_risk         0.04  ████
balance_to_salary_ratio       0.03  ██
low_point_earned_risk         0.01  █
young_explorer               -0.04  ████
num_of_products              -0.05  ████
geography_Spain              -0.05  █████
open_account_bf_majo

## 7. Sauvegarde du dataset enrichi

Export du dataset avec les nouvelles features pour la phase de modélisation.

**Résumé des transformations** :
- **4 flags de risque** créés
- **1 variable binned** (age)
- **3 interactions binaires** entre variables à fort churn
- **11 features métier** (ratios, segments comportementaux, scores d'engagement)

**Dataset final** : 10,000 observations × 32 features (dont 1 cible `exited`)

**Prochaine étape** : Notebook 04 - Modélisation et évaluation des performances

In [12]:
print("="*80)
print("Vérification avant sauvegarde")
print("="*80)

print(f"Dimensions du dataframe : {df.shape}")
print(f"Valeurs manquantes : {df.isna().sum().sum()}")
print("\nTypes des colonnes :") 
print(df.dtypes)

df.to_csv('../data/preprocessed/03_churn_records.csv', index=False)
print(f"\ncsv sauvegardé à l'emplacement '../data/preprocessed/03_churn_records.csv'")

Vérification avant sauvegarde
Dimensions du dataframe : (10000, 29)
Valeurs manquantes : 0

Types des colonnes :
age                            Int64
balance                      Float64
num_of_products                Int64
is_active_member               Int64
exited                         Int64
geography_France             boolean
geography_Germany            boolean
geography_Spain              boolean
gender_Female                boolean
gender_Male                  boolean
medium_balance_risk          boolean
low_credit_score_risk        boolean
low_point_earned_risk        boolean
open_account_bf_major        boolean
age_bin                     category
product_1_AND_germany        boolean
germany_AND_female           boolean
product_1_AND_female         boolean
product_1_inactive             int32
product_engagement_score       Int64
balance_to_salary_ratio      Float64
balance_per_product          Float64
high_balance_inactive          int32
young_explorer                 int32